# Implied Monetary Policy Pricer — Global EM

Backs the market-implied central bank policy path out of live curves, **per meeting date**, for
Latam (Mexico, Chile, Colombia), CE3 (Poland, Hungary, Turkey) and Asia (India, Indonesia, Thailand,
Malaysia, Philippines) — replacing fixed-horizon MIPR-style output with something whose assumptions
you can see and change.

**Run All** to launch. All data is pulled live from Bloomberg via BQL at refresh time — there are no
hardcoded market snapshots anywhere in this notebook.

What you get, top to bottom on one page:

1. **Implied path chart** — the policy rate the market prices after each meeting, plus WIRP-style
   bars showing the number of standard moves priced per meeting.
2. **Curve check** — the live curve quotes, the smooth interpolated curve, and the curve *rebuilt
   from the fitted meeting path*, so you can verify the implied path actually explains the market.
3. **Meeting table** — moves in bp, # of standard moves, and the implied rate after each meeting.
4. **Scenario & fair value** — replace the market's path with your own meeting-by-meeting view and
   see which curve instruments look rich or cheap against it, in bp.

Every market lets you **choose the curve** when more than one is available (e.g. Mexico: current
TIIE-F RFR vs legacy TIIE 28D), and the assumptions the path depends on — policy anchor, O/N anchor,
curve-to-policy basis, standard move size, interpolation, path smoothing, meeting calendar — are all
on screen and editable. The full methodology is at the bottom of the notebook and inside the app.


In [ ]:
# =============================================================================
# SETUP
# =============================================================================
# This tool is live-data only: every curve, policy rate and (where available)
# meeting calendar is pulled from Bloomberg via BQL at refresh time. Run it
# inside BQuant with Run All.

from __future__ import annotations

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from scipy.interpolate import PchipInterpolator, interp1d
from scipy.optimize import least_squares

import plotly.graph_objects as go
from plotly.subplots import make_subplots

import ipywidgets as widgets
from IPython.display import HTML, Markdown, clear_output, display

try:
    import bql
    BQ = bql.Service()
    HAS_BQL = True
except Exception:
    bql, BQ, HAS_BQL = None, None, False

if not HAS_BQL:
    display(HTML(
        "<div style='border:1px solid #b45309; padding:10px 14px; border-radius:6px;'>"
        "<b>No BQL session detected.</b> This tool pulls all data live from "
        "Bloomberg and must be run inside BQuant. Open it in BQuant and click "
        "Run All.</div>"
    ))


In [ ]:
# =============================================================================
# 1. MARKETS IN SCOPE
# =============================================================================
# One entry per market. Each market can offer MORE THAN ONE curve -- the
# dropdown in the app lets the user pick which curve the implied path is
# derived from, and each curve option carries its own basis assumption and a
# note on liquidity/status. Curve numbers (YCSW0xxx Index = ICVS curve xxx)
# come from the desk's own notes; confirm any curve on the terminal via ICVS
# or CRVF before production use.
#
# Mexico is the instructive example for "be smart about curve choice":
# the legacy 28-day TIIE swap was historically the liquid MXN curve, but
# Banxico's benchmark transition converted cleared 28D TIIE swaps to the
# overnight F-TIIE (RFR) during 2024 and new 28D fixings were only allowed
# under a waiver through end-2025. As of 2026 liquidity is in the TIIE-F
# RFR curve, so that is the default -- the legacy curve remains selectable
# for comparison, with this caveat attached where the user can see it.

CURVE_LIBRARY = {
    "Mexico": [
        {"label": "TIIE-F RFR OIS (S0583) — current liquid curve", "ticker": "YCSW0583 Index",
         "basis_bps": 0.0,
         "note": "F-TIIE overnight OIS. The liquid MXN curve since Banxico's 2024-25 "
                 "benchmark transition away from 28D TIIE."},
        {"label": "Legacy TIIE 28D IRS (S0083)", "ticker": "YCSW0083 Index",
         "basis_bps": 25.0,
         "note": "Legacy 28-day TIIE swap curve. Was the most liquid MXN curve "
                 "historically, but new 28D TIIE fixings ended with the post-2025 "
                 "conversion waiver -- quotes may be stale/synthetic. Basis default "
                 "+25bp reflects the historical TIIE28-to-target spread; verify."},
    ],
    "Chile": [
        {"label": "CLP Cámara OIS (S0193)", "ticker": "YCSW0193 Index", "basis_bps": 0.0,
         "note": "Chile has no liquid short futures/FRAs; the Cámara OIS curve is the market standard."},
    ],
    "Colombia": [
        {"label": "COP IBR OIS (S0329)", "ticker": "YCSW0329 Index", "basis_bps": 0.0,
         "note": "IBR overnight OIS curve."},
    ],
    "Poland": [
        {"label": "PLN WIBOR IRS (S0327)", "ticker": "YCSW0327 Index", "basis_bps": 0.0,
         "note": "WIBOR-referencing IRS curve (desk notes: curve 327). WIBOR fixes at a "
                 "spread to the NBP reference rate -- set the basis input accordingly if "
                 "you want the path in policy-rate terms."},
    ],
    "Hungary": [
        {"label": "HUF BUBOR IRS (S0324)", "ticker": "YCSW0324 Index", "basis_bps": 0.0,
         "note": "BUBOR-referencing IRS curve (desk notes: curve 324, flagged 'confirm')."},
    ],
    "Turkey": [
        {"label": "TRY OIS (S0522)", "ticker": "YCSW0522 Index", "basis_bps": 300.0,
         "note": "TLREF OIS. The effective overnight rate trades well above the 1W repo "
                 "policy rate -- default basis +300bp translates the O/N path into "
                 "policy-equivalent terms; adjust to the current corridor position."},
    ],
    "India": [
        {"label": "INR MIBOR OIS (S0266)", "ticker": "YCSW0266 Index", "basis_bps": 0.0,
         "note": "Overnight MIBOR OIS curve (desk notes: curve 266)."},
    ],
    "Indonesia": [
        {"label": "IDR OIS (S0158)", "ticker": "YCSW0158 Index", "basis_bps": 0.0,
         "note": "Desk notes flag this curve number 'confirm?'. Verify via ICVS before use."},
    ],
    "Thailand": [
        {"label": "THB THOR OIS (S0552)", "ticker": "YCSW0552 Index", "basis_bps": 0.0,
         "note": "THOR OIS curve (desk notes: curve 552)."},
    ],
    "Malaysia": [
        {"label": "MYR KLIBOR IRS (S0209)", "ticker": "YCSW0209 Index", "basis_bps": 0.0,
         "note": "KLIBOR-referencing IRS curve (desk notes: curve 209). KLIBOR 3M fixes "
                 "at a spread to the OPR -- adjust the basis input if needed."},
    ],
    "Philippines": [
        {"label": "PHP swap curve (S0370)", "ticker": "YCSW0370 Index", "basis_bps": 0.0,
         "note": "Desk notes flag this curve number 'confirm?'. Verify via ICVS before use."},
    ],
}

# Policy-rate tickers (used to anchor the path; pulled live, editable in the UI).
POLICY_TICKERS = {
    "Mexico": "MXONBR Index",
    "Chile": "CHOVCHOV Index",
    "Colombia": "CORRMIN Index",
    "Poland": "POREANN Index",
    "Hungary": "HBBRDEPO Index",
    "Turkey": "TUBR1WRA Index",
    "India": "INRPYLDP Index",
    "Indonesia": "IDBIRATE Index",
    "Thailand": "BTRR1DAY Index",
    "Malaysia": "MAOPRATE Index",
    "Philippines": "PPCBON Index",
}

# Day-count basis per market (simple ACT/basis conventions for the <=13M
# bullet approximation used throughout; exposed in the Assumptions panel).
DAY_COUNT = {
    "Mexico": 360, "Chile": 360, "Colombia": 360, "Poland": 365,
    "Hungary": 360, "Turkey": 360, "India": 365, "Indonesia": 360,
    "Thailand": 365, "Malaysia": 365, "Philippines": 360,
}

# Standard policy move size per market, in bp (drives the "# moves priced" units).
STANDARD_MOVE = {
    "Mexico": 25.0, "Chile": 25.0, "Colombia": 50.0, "Poland": 25.0,
    "Hungary": 25.0, "Turkey": 250.0, "India": 25.0, "Indonesia": 25.0,
    "Thailand": 25.0, "Malaysia": 25.0, "Philippines": 25.0,
}

# Bounds on the per-meeting move the calibration may assign (bp) -- wide for
# markets that have moved in big steps recently.
MOVE_BOUNDS = {
    "Mexico": (-100, 100), "Chile": (-100, 100), "Colombia": (-300, 300),
    "Poland": (-100, 100), "Hungary": (-150, 100), "Turkey": (-500, 250),
    "India": (-100, 100), "Indonesia": (-100, 100), "Thailand": (-75, 75),
    "Malaysia": (-75, 75), "Philippines": (-100, 100),
}

# Country universes for the live BQL central-bank calendar pull.
CALENDAR_UNIVERSE = {
    "Mexico": "MX Country", "Chile": "CL Country", "Colombia": "CO Country",
    "Poland": "PL Country", "Hungary": "HU Country", "Turkey": "TR Country",
    "India": "IN Country", "Indonesia": "ID Country", "Thailand": "TH Country",
    "Malaysia": "MY Country", "Philippines": "PH Country",
}

# Static fallback meeting dates, used only if the live calendar pull fails.
# These reflect published central-bank schedules where available; countries
# whose official calendar was not confirmed are cadence estimates -- the app
# labels which source it actually used.
FALLBACK_MEETINGS = {
    "Mexico": ["2026-08-06", "2026-09-24", "2026-11-05", "2026-12-17"],
    "Chile": ["2026-07-28", "2026-09-08", "2026-10-27", "2026-12-15"],
    "Colombia": ["2026-07-31", "2026-09-30", "2026-10-30", "2026-12-18"],
    "Poland": ["2026-09-02", "2026-10-07", "2026-11-04", "2026-12-02"],
    "Hungary": ["2026-07-28", "2026-08-25", "2026-09-22", "2026-10-27", "2026-11-24", "2026-12-15"],
    "Turkey": ["2026-07-23", "2026-09-10", "2026-10-22", "2026-12-10"],
    "India": ["2026-08-05", "2026-10-07", "2026-12-04"],
    "Indonesia": ["2026-07-22", "2026-08-19", "2026-09-23", "2026-10-21", "2026-11-18", "2026-12-16"],
    "Thailand": ["2026-08-19", "2026-10-14", "2026-12-09"],
    "Malaysia": ["2026-09-03", "2026-11-05"],
    "Philippines": ["2026-07-30", "2026-09-10", "2026-10-22", "2026-12-03"],
}

REGION = {
    "Mexico": "Latam", "Chile": "Latam", "Colombia": "Latam",
    "Poland": "CE3", "Hungary": "CE3", "Turkey": "CE3",
    "India": "Asia", "Indonesia": "Asia", "Thailand": "Asia",
    "Malaysia": "Asia", "Philippines": "Asia",
}

# Africa: no tickers, curve numbers or calendars existed in the source
# material, so no African market is configured yet. Adding one (e.g. South
# Africa) means adding entries to the dicts above -- the engine and UI below
# are fully country-agnostic.

print(f"{len(CURVE_LIBRARY)} markets configured "
      f"({sum(len(v) for v in CURVE_LIBRARY.values())} curve options).")


In [ ]:
# =============================================================================
# 2. LIVE DATA LAYER
# =============================================================================
# Everything here is pulled live at refresh time. Each pull is isolated with
# its own clear error/fallback behavior so one bad ticker degrades gracefully
# instead of taking the whole app down (the app reports exactly what was used).

def _pick_col(df, candidates):
    """Finds a column by exact (case-insensitive) match first, substring second."""
    lower = {str(c).lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower:
            return lower[cand.lower()]
    for col in df.columns:
        for cand in candidates:
            if cand.lower() in str(col).lower():
                return col
    return None


def fetch_curve(curve_ticker: str, as_of: pd.Timestamp, max_days: int = 430) -> pd.DataFrame:
    """Pulls all members of a Bloomberg curve object, keeps those maturing
    within `max_days`, and returns tidy nodes: ticker / rate / maturity / days."""
    universe = BQ.univ.curvemembers(symbols=curve_ticker)
    raw = BQ.execute(bql.Request(universe, {"curve_rate": BQ.data.curve_rate(side="MID")}))[0].df().reset_index()

    id_col = _pick_col(raw, ["ID", "index", "security"]) or raw.columns[0]
    rate_col = _pick_col(raw, ["curve_rate", "rate", "value"])
    if rate_col is None:
        raise RuntimeError(f"{curve_ticker}: no rate column in curve response ({list(raw.columns)}).")

    members = pd.DataFrame({
        "ticker": raw[id_col].astype(str),
        "rate": pd.to_numeric(raw[rate_col], errors="coerce"),
    }).dropna(subset=["rate"])
    members = members[(members["rate"] > -20) & (members["rate"] < 200)]
    if members.empty:
        raise RuntimeError(f"{curve_ticker}: curve returned no usable rates.")

    mat_raw = BQ.execute(bql.Request(members["ticker"].tolist(), {"maturity": BQ.data.maturity()}))[0].df().reset_index()
    mat_id = _pick_col(mat_raw, ["ID", "index", "security"]) or mat_raw.columns[0]
    mat_col = _pick_col(mat_raw, ["maturity"])
    maturities = pd.DataFrame({
        "ticker": mat_raw[mat_id].astype(str),
        "maturity": pd.to_datetime(mat_raw[mat_col], errors="coerce"),
    })

    nodes = members.merge(maturities, on="ticker", how="left").dropna(subset=["maturity"])
    nodes["days"] = (nodes["maturity"] - as_of).dt.days
    nodes = nodes[(nodes["days"] >= 3) & (nodes["days"] <= max_days)]
    nodes = nodes.sort_values("days").drop_duplicates("days", keep="first").reset_index(drop=True)

    if len(nodes) < 3:
        raise RuntimeError(
            f"{curve_ticker}: only {len(nodes)} node(s) within {max_days} days -- "
            "not enough short-end points to imply a meeting path. Try another "
            "curve option for this market, or confirm the curve number via ICVS."
        )
    return nodes


def fetch_policy_rate(country: str) -> "tuple[float | None, str]":
    """Returns (rate, source_note). Falls back to None (caller anchors off the
    curve front end) rather than raising."""
    ticker = POLICY_TICKERS.get(country)
    if not ticker:
        return None, "no policy ticker configured"
    try:
        raw = BQ.execute(bql.Request([ticker], {"px": BQ.data.px_last()}))[0].df().reset_index()
        col = _pick_col(raw, ["px", "px_last", "value"]) or raw.columns[-1]
        value = pd.to_numeric(raw[col], errors="coerce").iloc[0]
        if pd.notna(value) and -5 < float(value) < 200:
            return float(value), f"live ({ticker})"
    except Exception as exc:
        return None, f"pull failed ({ticker}: {exc})"
    return None, f"non-numeric ({ticker})"


def fetch_meetings(country: str, as_of: pd.Timestamp, horizon_days: int = 400) -> "tuple[list, str]":
    """Returns (future_meeting_dates, source_note). Tries the live Bloomberg
    central-bank calendar first; falls back to the static schedule."""
    fallback = [pd.Timestamp(d) for d in FALLBACK_MEETINGS[country]]
    fallback = [d for d in fallback if d > as_of]

    universe = CALENDAR_UNIVERSE.get(country)
    if universe and HAS_BQL:
        try:
            item = BQ.data.calendar(
                type="CENTRAL_BANKS", subtype="ALL_CENTRAL_BANKS",
                dates=BQ.func.range(
                    as_of.strftime("%Y-%m-%d"),
                    (as_of + pd.Timedelta(days=horizon_days)).strftime("%Y-%m-%d"),
                ),
                view="EXTENDED",
            )
            raw = BQ.execute(bql.Request(universe, {"cal": item}))[0].df().reset_index()
            date_col = _pick_col(raw, ["RELEASE_DATE", "release_date"])
            name_col = _pick_col(raw, ["EVENT_NAME", "event_name"])
            if date_col and name_col:
                df = raw[[date_col, name_col]].copy()
                df.columns = ["date", "event"]
                df["date"] = pd.to_datetime(df["date"], errors="coerce")
                df = df.dropna(subset=["date"])
                event = df["event"].astype(str).str.lower()
                keep = event.str.contains("rate decision|policy rate|overnight rate|target rate|reference rate|base rate|repo rate|bi rate|opr|rrp", regex=True)
                drop = event.str.contains("minutes|speech|conference|forecast|report", regex=True)
                dates = sorted(df.loc[keep & ~drop, "date"].dt.normalize().unique())
                dates = [pd.Timestamp(d) for d in dates if pd.Timestamp(d) > as_of]
                if len(dates) >= 2:
                    return dates[:8], "live Bloomberg central-bank calendar"
        except Exception:
            pass

    return fallback[:8], "static fallback schedule (live calendar unavailable)"


In [ ]:
# =============================================================================
# 3. CURVE + MEETING-PATH ENGINE
# =============================================================================
# Three steps, matching the methodology section at the bottom of the notebook:
#
#   (1) Quoted short-end rates -> discount factors -> smooth curve by
#       interpolating LOG discount factors (PCHIP by default; linear
#       selectable). Never interpolate raw rates: on a concave curve that
#       systematically understates forwards.
#
#   (2) The policy path is piecewise-constant, jumping only on meeting dates.
#       The per-meeting jumps are solved by least squares so that repricing
#       every quoted node under the step path matches the market curve --
#       i.e. the meeting path is BACKED OUT of the curve, not assumed.
#
#   (3) A user scenario is just a replacement set of jumps; repricing the
#       same nodes under it gives scenario-fair rates, and the gap to market
#       is the "fair value" of the view in bp.


def build_smooth_curve(nodes: pd.DataFrame, basis: int, method: str):
    """nodes: columns rate (%, par bullet) and days. Returns logdf_fn."""
    alpha = nodes["days"].to_numpy(float) / basis
    dfs = 1.0 / (1.0 + nodes["rate"].to_numpy(float) / 100.0 * alpha)
    x = np.r_[0.0, nodes["days"].to_numpy(float)]
    y = np.r_[0.0, np.log(dfs)]
    if method.startswith("Linear"):
        return interp1d(x, y, fill_value="extrapolate")
    return PchipInterpolator(x, y, extrapolate=True)


def par_from_logdf(logdf_fn, days: float, basis: int) -> float:
    df = float(np.exp(logdf_fn(float(days))))
    alpha = days / basis
    return ((1.0 / df - 1.0) / alpha) * 100.0


def step_df(days: float, boundaries: np.ndarray, seg_rates_pct: np.ndarray, basis: int) -> float:
    """Discount factor to `days` under a piecewise-constant O/N path with
    daily compounding; rates change only at boundary days (= meetings)."""
    acc = 1.0
    for i, rate in enumerate(seg_rates_pct):
        start, end = int(boundaries[i]), int(boundaries[i + 1])
        overlap = max(min(int(days), end) - start, 0)
        if overlap > 0:
            acc *= (1.0 + float(rate) / 100.0 / basis) ** overlap
        if days <= end:
            break
    return 1.0 / acc


def par_from_steps(days: float, boundaries, seg_rates_pct, basis: int) -> float:
    df = step_df(days, boundaries, seg_rates_pct, basis)
    alpha = days / basis
    return ((1.0 / df - 1.0) / alpha) * 100.0


def calibrate_meeting_path(nodes: pd.DataFrame, meeting_days: "list[int]",
                           on_anchor: float, policy_anchor: float, basis_bps: float,
                           day_count: int, move_bounds: "tuple", standard_move: float,
                           smoothness: float = 0.10):
    """Solves the per-meeting policy jumps (bp) that reprice the curve nodes.

    Returns (moves_bps, seg_rates, boundaries, fit_table).
    seg_rates are O/N-equivalent segment rates: [anchor, post-mtg-1, ...].
    """
    n = len(meeting_days)
    horizon = int(nodes["days"].max()) + 30
    boundaries = np.asarray([0] + list(meeting_days) + [horizon], dtype=int)

    target_days = nodes["days"].to_numpy(float)
    target_rates = nodes["rate"].to_numpy(float)
    weights = 1.0 / np.sqrt(np.maximum(target_days, 7.0) / 7.0)

    total_gap_bps = (target_rates[-1] - on_anchor) * 100.0
    x0 = np.clip(np.full(n, total_gap_bps / max(n, 1)), move_bounds[0], move_bounds[1])

    def seg_rates_from(moves_bps):
        policy = policy_anchor + np.cumsum(moves_bps) / 100.0
        return np.r_[on_anchor, policy + basis_bps / 100.0]

    def residuals(moves_bps):
        rates = seg_rates_from(moves_bps)
        errs = [
            (par_from_steps(d, boundaries, rates, day_count) - r) * 100.0 * w
            for d, r, w in zip(target_days, target_rates, weights)
        ]
        smooth = np.sqrt(smoothness) * np.diff(moves_bps) / standard_move if n > 1 else np.array([])
        size = np.sqrt(0.02) * moves_bps / standard_move
        return np.r_[errs, smooth, size]

    result = least_squares(
        residuals, x0,
        bounds=(np.full(n, float(move_bounds[0])), np.full(n, float(move_bounds[1]))),
        xtol=1e-11, ftol=1e-11, gtol=1e-11, max_nfev=20000,
    )
    if not result.success:
        raise RuntimeError(f"Meeting-path calibration failed: {result.message}")

    moves = result.x
    seg_rates = seg_rates_from(moves)

    fit = nodes.copy()
    fit["model_rate"] = [par_from_steps(d, boundaries, seg_rates, day_count) for d in target_days]
    fit["error_bps"] = (fit["model_rate"] - fit["rate"]) * 100.0
    return moves, seg_rates, boundaries, fit


def scenario_fair_value(nodes: pd.DataFrame, boundaries, day_count: int,
                        on_anchor: float, policy_anchor: float, basis_bps: float,
                        scenario_moves_bps) -> pd.DataFrame:
    """Reprices every curve node under the user's meeting path."""
    policy = policy_anchor + np.cumsum(np.asarray(scenario_moves_bps, float)) / 100.0
    seg_rates = np.r_[on_anchor, policy + basis_bps / 100.0]
    out = nodes[["ticker", "maturity", "days", "rate"]].copy()
    out = out.rename(columns={"rate": "market_rate"})
    out["scenario_rate"] = [
        par_from_steps(d, boundaries, seg_rates, day_count) for d in out["days"]
    ]
    out["gap_bps"] = (out["scenario_rate"] - out["market_rate"]) * 100.0
    return out


In [ ]:
# =============================================================================
# 4. APP
# =============================================================================
# One scrolling page: pick country + curve -> implied path chart -> backed-out
# curve check -> meeting table -> scenario + fair value. Assumptions and the
# methodology live in a collapsed accordion at the bottom.

BLUE, ORANGE, PINK, GREY = "#2f81f7", "#f0883e", "#db61a2", "#8b949e"


def _summary_table(rows):
    """Plain two-column HTML table -- deliberately simple so it renders
    identically in BQuant, Jupyter and VS Code with no overlap."""
    tr = "".join(
        f"<tr><td style='padding:2px 14px 2px 0; color:#8b949e; white-space:nowrap;'>{k}</td>"
        f"<td style='padding:2px 0;'>{v}</td></tr>"
        for k, v in rows
    )
    return f"<table style='border-collapse:collapse; font-size:13px;'>{tr}</table>"


class PolicyPricerApp:
    def __init__(self):
        self.as_of = pd.Timestamp.today().normalize()
        self.nodes = None
        self.logdf_fn = None
        self.moves = None
        self.seg_rates = None
        self.boundaries = None
        self.fit = None
        self.meetings = []
        self.meeting_days = []
        self.calendar_note = ""
        self.policy_note = ""
        self.move_inputs = []
        self._loading = False

        self._build_widgets()
        self._build_layout()

    # -- widgets ---------------------------------------------------------
    def _build_widgets(self):
        countries = sorted(CURVE_LIBRARY, key=lambda c: (REGION[c], c))
        self.country = widgets.Dropdown(
            options=[(f"{REGION[c]} · {c}", c) for c in countries], value="Mexico",
            description="Country:", layout=widgets.Layout(width="270px"))
        self.curve_choice = widgets.Dropdown(
            options=[(o["label"], i) for i, o in enumerate(CURVE_LIBRARY["Mexico"])],
            value=0, description="Curve:", layout=widgets.Layout(width="430px"))
        self.refresh_btn = widgets.Button(description="Refresh live data", button_style="primary", icon="refresh")

        self.policy_in = widgets.FloatText(description="Policy rate %:", layout=widgets.Layout(width="200px"))
        self.anchor_in = widgets.FloatText(description="O/N anchor %:", layout=widgets.Layout(width="200px"))
        self.basis_in = widgets.FloatText(description="Basis bp:", layout=widgets.Layout(width="180px"))
        self.stdmove_in = widgets.FloatText(description="Std move bp:", layout=widgets.Layout(width="190px"))

        self.interp_in = widgets.Dropdown(
            options=["PCHIP on log discount factors (recommended)", "Linear on log discount factors"],
            description="Interpolation:", style={"description_width": "110px"},
            layout=widgets.Layout(width="440px"))
        self.smooth_in = widgets.FloatText(value=0.10, description="Path smoothing λ:",
                                           style={"description_width": "130px"},
                                           layout=widgets.Layout(width="240px"))

        self.preset_in = widgets.Dropdown(
            options=["Market-implied", "Hold at current rate", "Cut 1 std move each meeting",
                     "Hike 1 std move each meeting"],
            description="Preset:", layout=widgets.Layout(width="330px"))
        self.apply_btn = widgets.Button(description="Apply preset", icon="magic")
        self.scenario_btn = widgets.Button(description="Reprice scenario", button_style="success", icon="play")

        self.summary_out = widgets.Output()
        self.chart_out = widgets.Output()
        self.curve_out = widgets.Output()
        self.table_out = widgets.Output()
        self.scenario_inputs_out = widgets.Output()
        self.scenario_out = widgets.Output()
        self.assumptions_out = widgets.Output()
        self.methodology_out = widgets.Output()

        self.country.observe(self._on_country, names="value")
        self.curve_choice.observe(self._on_curve, names="value")
        self.refresh_btn.on_click(lambda _: self.refresh())
        self.apply_btn.on_click(lambda _: self._apply_preset())
        self.scenario_btn.on_click(lambda _: self.render_scenario())
        self.interp_in.observe(self._on_curve, names="value")

    def _build_layout(self):
        advanced = widgets.VBox([
            widgets.HTML("<div style='color:#8b949e; margin:4px 0;'>Model choices — every knob "
                         "the implied path depends on. Changing interpolation recalibrates.</div>"),
            widgets.HBox([self.interp_in, self.smooth_in]),
            self.assumptions_out,
        ])
        self.accordion = widgets.Accordion(children=[advanced, self.methodology_out])
        self.accordion.set_title(0, "Assumptions & diagnostics")
        self.accordion.set_title(1, "Methodology")
        self.accordion.selected_index = None

        self.page = widgets.VBox([
            widgets.HTML("<h2 style='margin:2px 0 0;'>Implied Monetary Policy Pricer — Global EM</h2>"
                         "<div style='color:#8b949e; margin:2px 0 10px;'>Meeting-by-meeting policy "
                         "pricing backed out of live curves. Pick a market and a curve; every "
                         "assumption is visible and editable.</div>"),
            widgets.HBox([self.country, self.curve_choice, self.refresh_btn]),
            widgets.HBox([self.policy_in, self.anchor_in, self.basis_in, self.stdmove_in]),
            self.summary_out,
            self.chart_out,
            self.curve_out,
            self.table_out,
            widgets.HTML("<h3 style='margin:16px 0 2px;'>Scenario — what if the central bank does this instead?</h3>"
                         "<div style='color:#8b949e; margin:0 0 6px;'>Enter a move in bp at each "
                         "meeting (−25 = cut, 0 = hold, +25 = hike), then reprice. Fair value = what "
                         "each curve instrument should yield under your path vs where it trades.</div>"),
            widgets.HBox([self.preset_in, self.apply_btn, self.scenario_btn]),
            self.scenario_inputs_out,
            self.scenario_out,
            self.accordion,
        ])

    def display(self):
        display(self.page)

    # -- config helpers --------------------------------------------------
    def _curve_option(self):
        return CURVE_LIBRARY[self.country.value][self.curve_choice.value]

    def _on_country(self, change):
        if self._loading:
            return
        self._loading = True
        options = CURVE_LIBRARY[change["new"]]
        self.curve_choice.options = [(o["label"], i) for i, o in enumerate(options)]
        self.curve_choice.value = 0
        self.stdmove_in.value = STANDARD_MOVE[change["new"]]
        self.basis_in.value = options[0]["basis_bps"]
        self._loading = False
        self.refresh()

    def _on_curve(self, change):
        if self._loading:
            return
        self.basis_in.value = self._curve_option()["basis_bps"]
        self.refresh()

    # -- data + calibration ----------------------------------------------
    def refresh(self):
        country = self.country.value
        option = self._curve_option()
        with self.summary_out:
            clear_output()
            display(HTML("<i>Pulling live data and calibrating…</i>"))
        try:
            if not HAS_BQL:
                raise RuntimeError("No BQL session — run this notebook inside BQuant.")

            self.nodes = fetch_curve(option["ticker"], self.as_of)
            self.logdf_fn = build_smooth_curve(self.nodes, DAY_COUNT[country], self.interp_in.value)

            policy, self.policy_note = fetch_policy_rate(country)
            front = float(self.nodes["rate"].iloc[0])
            if policy is None:
                policy = front - float(self.basis_in.value) / 100.0
                self.policy_note += " → anchored off curve front end"
            self.policy_in.value = round(policy, 4)
            self.anchor_in.value = round(front, 4)
            if self.stdmove_in.value == 0:
                self.stdmove_in.value = STANDARD_MOVE[country]

            self.meetings, self.calendar_note = fetch_meetings(country, self.as_of)
            max_priceable = int(self.nodes["days"].max()) - 7
            self.meetings = [m for m in self.meetings
                             if 0 < (m - self.as_of).days <= max_priceable]
            if len(self.meetings) < 1:
                raise RuntimeError("No future meetings fall inside the curve's horizon.")
            self.meeting_days = [(m - self.as_of).days for m in self.meetings]

            self.moves, self.seg_rates, self.boundaries, self.fit = calibrate_meeting_path(
                self.nodes, self.meeting_days,
                on_anchor=float(self.anchor_in.value),
                policy_anchor=float(self.policy_in.value),
                basis_bps=float(self.basis_in.value),
                day_count=DAY_COUNT[country],
                move_bounds=MOVE_BOUNDS[country],
                standard_move=float(self.stdmove_in.value),
                smoothness=max(float(self.smooth_in.value), 0.0),
            )

            self._build_move_inputs()
            self.render_summary()
            self.render_main_chart()
            self.render_curve_chart()
            self.render_table()
            self.render_scenario()
            self.render_assumptions()
            self.render_methodology()

        except Exception as exc:
            with self.summary_out:
                clear_output()
                display(HTML(f"<div style='color:#f85149;'><b>Could not price {country}:</b> {exc}</div>"))

    # -- scenario helpers --------------------------------------------------
    def _build_move_inputs(self):
        self.move_inputs = [
            widgets.FloatText(value=round(float(m), 1), description=d.strftime("%d %b %y"),
                              layout=widgets.Layout(width="215px"))
            for d, m in zip(self.meetings, self.moves)
        ]
        with self.scenario_inputs_out:
            clear_output()
            display(widgets.GridBox(children=self.move_inputs,
                                    layout=widgets.Layout(grid_template_columns="repeat(4, 225px)",
                                                          grid_gap="4px 12px")))

    def _apply_preset(self):
        preset, std = self.preset_in.value, float(self.stdmove_in.value)
        if preset == "Market-implied":
            values = [round(float(m), 1) for m in self.moves]
        elif preset == "Hold at current rate":
            values = [0.0] * len(self.move_inputs)
        elif preset.startswith("Cut"):
            values = [-abs(std)] * len(self.move_inputs)
        else:
            values = [abs(std)] * len(self.move_inputs)
        for w, v in zip(self.move_inputs, values):
            w.value = v
        self.render_scenario()

    def _user_moves(self):
        return [float(w.value) for w in self.move_inputs]

    # -- rendering ---------------------------------------------------------
    def render_summary(self):
        option = self._curve_option()
        std = float(self.stdmove_in.value)
        first_move = float(self.moves[0])
        total = float(np.sum(self.moves))
        rmse = float(np.sqrt(np.mean(self.fit["error_bps"] ** 2)))
        rows = [
            ("As of", f"{self.as_of.date()}  ·  curve: {option['ticker']}  ·  all data live"),
            ("Policy rate", f"{self.policy_in.value:.2f}%  ({self.policy_note})"),
            ("Next meeting", f"{self.meetings[0].date()}  ·  market prices {first_move:+.0f} bp "
                             f"({first_move / std:+.2f} × {std:.0f}bp moves)"),
            ("Total priced", f"{total:+.0f} bp across {len(self.meetings)} meetings "
                             f"(through {self.meetings[-1].date()})"),
            ("Curve fit", f"{rmse:.1f} bp RMSE across {len(self.fit)} live nodes"),
            ("Meeting calendar", self.calendar_note),
        ]
        with self.summary_out:
            clear_output()
            display(HTML(_summary_table(rows)))

    def _step_xy(self, levels, start_level):
        x, y = [self.as_of], [start_level]
        prev = start_level
        for d, lv in zip(self.meetings, levels):
            x += [d, d]
            y += [prev, float(lv)]
            prev = float(lv)
        x.append(self.meetings[-1] + pd.Timedelta(days=21))
        y.append(prev)
        return x, y

    def render_main_chart(self):
        std = float(self.stdmove_in.value)
        policy_path = self.policy_in.value + np.cumsum(self.moves) / 100.0
        n_moves = self.moves / std

        fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                            row_heights=[0.7, 0.3], vertical_spacing=0.05)
        x, y = self._step_xy(policy_path, float(self.policy_in.value))
        fig.add_trace(go.Scatter(x=x, y=y, mode="lines", name="Market-implied policy rate",
                                 line={"width": 3, "color": BLUE},
                                 hovertemplate="%{x|%d %b %y}: %{y:.3f}%<extra></extra>"), row=1, col=1)
        fig.add_trace(go.Scatter(x=self.meetings, y=policy_path, mode="markers", showlegend=False,
                                 marker={"size": 8, "color": BLUE},
                                 hovertemplate="%{x|%d %b %y}: %{y:.3f}%<extra></extra>"), row=1, col=1)

        user = np.asarray(self._user_moves()) if self.move_inputs else self.moves
        if not np.allclose(user, self.moves, atol=0.5):
            ux, uy = self._step_xy(self.policy_in.value + np.cumsum(user) / 100.0,
                                   float(self.policy_in.value))
            fig.add_trace(go.Scatter(x=ux, y=uy, mode="lines", name="Your scenario",
                                     line={"width": 2, "color": PINK, "dash": "dash"},
                                     hovertemplate="%{x|%d %b %y}: %{y:.3f}%<extra></extra>"), row=1, col=1)

        fig.add_trace(go.Bar(x=self.meetings, y=n_moves,
                             name=f"# {std:.0f}bp moves priced",
                             marker={"color": [ORANGE if v >= 0 else BLUE for v in n_moves]},
                             hovertemplate="%{x|%d %b %y}: %{y:+.2f} moves<extra></extra>"), row=2, col=1)

        fig.update_yaxes(title_text="Policy rate (%)", row=1, col=1)
        fig.update_yaxes(title_text="# moves", row=2, col=1)
        fig.update_layout(height=520, hovermode="x unified", template="plotly_dark",
                          bargap=0.5, margin=dict(t=50, r=20, l=60, b=20),
                          legend={"orientation": "h", "y": 1.06, "x": 0},
                          title=f"{self.country.value}: implied policy path by meeting date")
        with self.chart_out:
            clear_output()
            fig.show()

    def render_curve_chart(self):
        """The 'backed-out curve' check: market nodes vs the smooth interpolated
        curve vs the curve rebuilt from the fitted meeting-step path."""
        country = self.country.value
        basis = DAY_COUNT[country]
        dense = np.arange(5, int(self.nodes["days"].max()) + 1, 2)
        dates = [self.as_of + pd.Timedelta(days=int(d)) for d in dense]

        smooth = [par_from_logdf(self.logdf_fn, d, basis) for d in dense]
        rebuilt = [par_from_steps(d, self.boundaries, self.seg_rates, basis) for d in dense]
        anchor_rates = np.asarray([float(self.anchor_in.value)])
        no_change = [par_from_steps(d, np.asarray([0, 10_000]), anchor_rates, basis) for d in dense]

        fig = go.Figure()
        fig.add_trace(go.Scatter(x=dates, y=no_change, mode="lines", name="No change (hold forever)",
                                 line={"width": 1, "color": GREY, "dash": "dot"},
                                 hovertemplate="%{y:.3f}%<extra>no change</extra>"))
        fig.add_trace(go.Scatter(x=dates, y=smooth, mode="lines", name="Interpolated market curve",
                                 line={"width": 2, "color": ORANGE},
                                 hovertemplate="%{y:.3f}%<extra>market curve</extra>"))
        fig.add_trace(go.Scatter(x=dates, y=rebuilt, mode="lines", name="Backed-out from meeting path",
                                 line={"width": 2, "color": BLUE, "dash": "dash"},
                                 hovertemplate="%{y:.3f}%<extra>meeting-step model</extra>"))
        fig.add_trace(go.Scatter(
            x=self.nodes["maturity"], y=self.nodes["rate"], mode="markers", name="Live curve quotes",
            marker={"size": 9, "color": "#e6edf3", "symbol": "diamond"},
            text=self.nodes["ticker"], hovertemplate="%{text}<br>%{y:.3f}%<extra></extra>"))
        for m in self.meetings:
            fig.add_vline(x=m.strftime("%Y-%m-%d"), line_width=1, line_dash="dot", line_color=GREY)

        fig.update_layout(height=380, hovermode="x unified", template="plotly_dark",
                          margin=dict(t=50, r=20, l=60, b=20),
                          legend={"orientation": "h", "y": 1.08, "x": 0},
                          title="Curve check — the meeting-step path reprices the live curve "
                                "(dotted verticals = meetings)",
                          yaxis_title="Par rate (%)")
        with self.curve_out:
            clear_output()
            fig.show()

    def render_table(self):
        std = float(self.stdmove_in.value)
        table = pd.DataFrame({
            "Meeting": [d.date() for d in self.meetings],
            "Implied move (bp)": np.round(self.moves, 1),
            f"# {std:.0f}bp moves": np.round(self.moves / std, 2),
            "Direction": np.where(self.moves > 1, "Hike", np.where(self.moves < -1, "Cut", "Hold")),
            "Policy rate after (%)": np.round(self.policy_in.value + np.cumsum(self.moves) / 100.0, 3),
        })
        with self.table_out:
            clear_output()
            display(table)

    def render_scenario(self):
        if self.nodes is None or not self.move_inputs:
            return
        country = self.country.value
        fv = scenario_fair_value(
            self.nodes, self.boundaries, DAY_COUNT[country],
            float(self.anchor_in.value), float(self.policy_in.value),
            float(self.basis_in.value), self._user_moves(),
        )
        show = fv.copy()
        show["maturity"] = pd.to_datetime(show["maturity"]).dt.date
        show.columns = ["Instrument", "Maturity", "Days", "Market rate (%)",
                        "Fair rate under your path (%)", "Rich/cheap (bp)"]

        fig = go.Figure(go.Bar(
            x=[str(m) for m in show["Maturity"]], y=show["Rich/cheap (bp)"],
            marker={"color": [ORANGE if v >= 0 else BLUE for v in show["Rich/cheap (bp)"]]},
            hovertemplate="%{x}: %{y:+.1f} bp<extra></extra>"))
        fig.update_layout(height=260, template="plotly_dark",
                          margin=dict(t=45, r=20, l=60, b=20),
                          title="Fair value vs market under your scenario "
                                "(+ = your path implies a higher rate than the market quotes)",
                          yaxis_title="bp")

        with self.scenario_out:
            clear_output()
            display(show.round(3))
            fig.show()
        self.render_main_chart()

    def render_assumptions(self):
        country = self.country.value
        option = self._curve_option()
        rows = [
            ("Curve", f"{option['label']}  ·  {option['ticker']}"),
            ("Curve note", option["note"]),
            ("Instruments used", f"{len(self.nodes)} live curve members maturing within ~14 months"),
            ("Day count", f"ACT/{DAY_COUNT[country]} (bullet approximation for sub-13M tenors)"),
            ("Interpolation", self.interp_in.value),
            ("Policy anchor", f"{self.policy_in.value:.2f}%  ({self.policy_note})"),
            ("O/N anchor", f"{self.anchor_in.value:.2f}% (defaults to the front curve node; editable)"),
            ("Curve-to-policy basis", f"{self.basis_in.value:+.0f} bp (editable)"),
            ("Standard move", f"{self.stdmove_in.value:.0f} bp"),
            ("Meeting calendar", f"{self.calendar_note}  ·  {len(self.meetings)} meetings in horizon"),
            ("Path smoothing λ", f"{self.smooth_in.value:g} (regularizes under-determined buckets)"),
        ]
        with self.assumptions_out:
            clear_output()
            display(HTML(_summary_table(rows)))
            display(HTML("<b style='font-size:13px;'>Calibration fit (model vs each live quote)</b>"))
            fit = self.fit[["ticker", "maturity", "days", "rate", "model_rate", "error_bps"]].copy()
            fit["maturity"] = pd.to_datetime(fit["maturity"]).dt.date
            fit.columns = ["Instrument", "Maturity", "Days", "Market (%)", "Model (%)", "Error (bp)"]
            display(fit.round(3))

    def render_methodology(self):
        with self.methodology_out:
            clear_output()
            display(Markdown(METHODOLOGY_TEXT))


In [ ]:
# =============================================================================
# 5. LAUNCH
# =============================================================================

METHODOLOGY_TEXT = r"""
## Methodology

### The problem with fixed-horizon pricers

Bloomberg MIPR shows "1y implied = x%" without telling you which instruments were used, which
interpolation, which day count, whether policy is assumed to move only at meetings, or how the first
fixing is handled. This tool prices to **actual meeting dates** and puts every one of those choices
on screen, editable.

### 1. Choosing the curve (instruments per market)

Each market's dropdown lists the curve(s) this desk considers usable, with a liquidity/status note.
The default is always the curve where liquidity currently sits — e.g. Mexico defaults to the
**TIIE-F RFR OIS** curve rather than the legacy 28-day TIIE swap curve, because Banxico's benchmark
transition converted cleared 28D TIIE swaps to overnight F-TIIE during 2024–25 and new 28D fixings
have ceased; the legacy curve remains selectable for comparison, flagged as such. Everything in
scope is swap/OIS-based because that is what is liquid in these markets (Chile, for example, has no
liquid short-dated futures or FRAs at all — the Cámara OIS curve *is* the market). Bonds were
deliberately excluded: government yields mix credit/liquidity premia into the policy signal.

### 2. Building the curve

Each live curve member maturing within ~14 months is treated as a single-payment par instrument:

$$DF(T) = \frac{1}{1 + S(T)\,\alpha(T)}$$

with $\alpha$ the ACT/360 or ACT/365 day-count fraction by market. The curve is then interpolated on
**log discount factors** — never on raw rates, which on a concave curve systematically understates
forwards (the exact issue flagged in the project notes). Default is PCHIP (shape-preserving, no
spline overshoot); a linear-on-log-DF mode is selectable to see how much the smoothing choice matters.

### 3. Backing out the meeting path

The policy path is **piecewise-constant, jumping only on meeting dates**. The per-meeting jumps
$\Delta_1,\dots,\Delta_n$ (bp) are solved by least squares so that repricing every live curve node
under the step path matches the market:

$$\min_{\Delta} \;\sum_k w_k\big(\hat S_k(\Delta) - S_k^{mkt}\big)^2 \;+\;
\lambda\sum_i(\Delta_{i+1}-\Delta_i)^2$$

The small smoothness penalty $\lambda$ only matters when one instrument tenor spans several meetings
(the system is then under-determined); it is exposed in the Assumptions panel. The **Curve check**
chart shows the result: the dashed line is the curve *rebuilt from the fitted meeting path* — if it
sits on top of the live quotes, the implied path genuinely explains the market.

### 4. Reading the output (WIRP conventions)

- **Step line** — the implied policy rate after each meeting; flat between meetings by construction.
- **Bars** — the move at each meeting divided by the market's standard move size. **+0.20** = about
  20% of one hike priced; **−0.66** = about 66% of one cut priced.
- The O/N-to-policy **basis** (editable) translates the curve's overnight fixing into policy-rate
  terms — e.g. Turkey's effective overnight rate trades ~300bp above the 1W repo rate, and legacy
  TIIE 28D fixed ~25bp above the Banxico target.

### 5. Scenario → fair value

Your per-meeting moves replace the market's. Every live curve instrument is then repriced under your
path with the identical discount-factor machinery (no shortcut averaging), and

$$\text{rich/cheap (bp)} = 100\,\big(S^{scenario} - S^{market}\big)$$

per instrument tells you where your view disagrees with the market most — positive means your path
implies a higher rate than the instrument currently yields (it looks rich to you; a hawkish view vs
market pricing), negative the reverse.

### Known limitations

The bullet-par approximation is standard for sub-13-month tenors but ignores coupon schedules and
exact settlement conventions; curve numbers taken from desk notes flagged "confirm" (Indonesia,
Philippines, and legacy Mexico S0083) should be verified via ICVS; meeting calendars fall back to a
static schedule when the live Bloomberg calendar pull returns nothing, and the app states which
source it used. Africa is not yet configured — adding a market is one entry in each config dict.
"""

app = PolicyPricerApp()
app.display()
app.refresh()


## Methodology

### The problem with fixed-horizon pricers

Bloomberg MIPR shows "1y implied = x%" without telling you which instruments were used, which
interpolation, which day count, whether policy is assumed to move only at meetings, or how the first
fixing is handled. This tool prices to **actual meeting dates** and puts every one of those choices
on screen, editable.

### 1. Choosing the curve (instruments per market)

Each market's dropdown lists the curve(s) this desk considers usable, with a liquidity/status note.
The default is always the curve where liquidity currently sits — e.g. Mexico defaults to the
**TIIE-F RFR OIS** curve rather than the legacy 28-day TIIE swap curve, because Banxico's benchmark
transition converted cleared 28D TIIE swaps to overnight F-TIIE during 2024–25 and new 28D fixings
have ceased; the legacy curve remains selectable for comparison, flagged as such. Everything in
scope is swap/OIS-based because that is what is liquid in these markets (Chile, for example, has no
liquid short-dated futures or FRAs at all — the Cámara OIS curve *is* the market). Bonds were
deliberately excluded: government yields mix credit/liquidity premia into the policy signal.

### 2. Building the curve

Each live curve member maturing within ~14 months is treated as a single-payment par instrument:

$$DF(T) = \frac{1}{1 + S(T)\,\alpha(T)}$$

with $\alpha$ the ACT/360 or ACT/365 day-count fraction by market. The curve is then interpolated on
**log discount factors** — never on raw rates, which on a concave curve systematically understates
forwards (the exact issue flagged in the project notes). Default is PCHIP (shape-preserving, no
spline overshoot); a linear-on-log-DF mode is selectable to see how much the smoothing choice matters.

### 3. Backing out the meeting path

The policy path is **piecewise-constant, jumping only on meeting dates**. The per-meeting jumps
$\Delta_1,\dots,\Delta_n$ (bp) are solved by least squares so that repricing every live curve node
under the step path matches the market:

$$\min_{\Delta} \;\sum_k w_k\big(\hat S_k(\Delta) - S_k^{mkt}\big)^2 \;+\;
\lambda\sum_i(\Delta_{i+1}-\Delta_i)^2$$

The small smoothness penalty $\lambda$ only matters when one instrument tenor spans several meetings
(the system is then under-determined); it is exposed in the Assumptions panel. The **Curve check**
chart shows the result: the dashed line is the curve *rebuilt from the fitted meeting path* — if it
sits on top of the live quotes, the implied path genuinely explains the market.

### 4. Reading the output (WIRP conventions)

- **Step line** — the implied policy rate after each meeting; flat between meetings by construction.
- **Bars** — the move at each meeting divided by the market's standard move size. **+0.20** = about
  20% of one hike priced; **−0.66** = about 66% of one cut priced.
- The O/N-to-policy **basis** (editable) translates the curve's overnight fixing into policy-rate
  terms — e.g. Turkey's effective overnight rate trades ~300bp above the 1W repo rate, and legacy
  TIIE 28D fixed ~25bp above the Banxico target.

### 5. Scenario → fair value

Your per-meeting moves replace the market's. Every live curve instrument is then repriced under your
path with the identical discount-factor machinery (no shortcut averaging), and

$$\text{rich/cheap (bp)} = 100\,\big(S^{scenario} - S^{market}\big)$$

per instrument tells you where your view disagrees with the market most — positive means your path
implies a higher rate than the instrument currently yields (it looks rich to you; a hawkish view vs
market pricing), negative the reverse.

### Known limitations

The bullet-par approximation is standard for sub-13-month tenors but ignores coupon schedules and
exact settlement conventions; curve numbers taken from desk notes flagged "confirm" (Indonesia,
Philippines, and legacy Mexico S0083) should be verified via ICVS; meeting calendars fall back to a
static schedule when the live Bloomberg calendar pull returns nothing, and the app states which
source it used. Africa is not yet configured — adding a market is one entry in each config dict.
